# 05 — Ensemble CNN + MLP (extra)

O enunciado pede os resultados de **cada** classificador (03 e 04); isto é um extra.
As duas redes usam as **mesmas janelas** (`data/processed/windows.csv`), por isso as previsões alinham-se diretamente.

Por paciente: $\hat P = w\,\hat P_{CNN} + (1-w)\,\hat P_{MLP}$.
Para não escolher `w` nos mesmos dados em que se reporta, a escolha é encaixada: para cada fold k, `w` **e** os limiares
são escolhidos nos outros 4 folds e aplicados ao fold k.

Precisa de `results/mlp/` e `results/cnn/` (notebooks 03/04 ou `python -m src.mlp` / `python -m src.cnn`).

In [ ]:
import sys; sys.path.insert(0, '..')
import json
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.data import CLASSES
from src.datasets import load_processed
from src.evaluate import RESULTS_DIR, patient_scores, cv_report, tune_thresholds, predict_rule, confusion_df, save_oof, load_oof, summary
from src.metrics import report
pd.set_option('display.width', 140)
d = load_processed()

In [ ]:
AGG = 'max'
pats = {}
for name in ['mlp', 'cnn']:
    idx, oof = load_oof(name)
    pats[name] = patient_scores(d.windows.loc[idx], oof, how=AGG)
common = pats['mlp'].index.intersection(pats['cnn'].index)
p_mlp, p_cnn = pats['mlp'].loc[common], pats['cnn'].loc[common]
assert (p_mlp.fold == p_cnn.fold).all() and (p_mlp.murmur == p_cnn.murmur).all()
print(len(common), 'pacientes')

def blend(w):
    p = p_cnn.copy(); p[CLASSES] = w * p_cnn[CLASSES] + (1 - w) * p_mlp[CLASSES]; return p

## Sensibilidade a `w` (só para visualizar; a escolha de `w` é feita na secção seguinte)

In [ ]:
W = np.round(np.linspace(0, 1, 11), 2)
rows = []
for w in W:
    t = cv_report(blend(w))[0]
    rows.append({'w_cnn': w, 'WA média': t['WA (tuned)'].mean(), 'WA std': t['WA (tuned)'].std()})
sweep = pd.DataFrame(rows).set_index('w_cnn')
ax =sweep['WA média'].plot(marker='o', figsize=(7, 3.5)); ax.fill_between(sweep.index, sweep['WA média'] - sweep['WA std'], sweep['WA média'] + sweep['WA std'], alpha=.15)
ax.set(xlabel='peso da CNN (w)', ylabel='WA (CV)', title='0 = só MLP, 1 = só CNN'); sweep.round(3).T

## Avaliação encaixada (w e limiares escolhidos sem ver o fold avaliado)

In [ ]:
rows, pred = [], pd.Series(index=common, dtype=object)
for k in sorted(p_cnn.fold.unique()):
    best = max(((tune_thresholds(blend(w)[p_cnn.fold != k])[1], w) for w in W))
    w = best[1]; p = blend(w)
    t, _ = tune_thresholds(p[p.fold != k])
    pred[p.fold == k] = predict_rule(p[p.fold == k], *t)
    rows.append({'fold': k, 'w_cnn': w, 't_present': t[0], 't_unknown': t[1],
                 'WA': report(p.murmur[p.fold == k], pred[p.fold == k])['weighted_accuracy']})
table = pd.DataFrame(rows).set_index('fold'); display(table.round(3))
rep = report(p_cnn.murmur, pred)
print('WA ensemble: %.3f ± %.3f | pooled %.3f' % (table.WA.mean(), table.WA.std(), rep['weighted_accuracy']))
print('recall:', {c: round(v, 3) for c, v in rep['recall'].items()})
confusion_df(rep)

## Comparação (mesmo protocolo, valores lidos de `results/`)

In [ ]:
rows = []
for name in ['mlp', 'cnn']:
    t, _, r = cv_report(pats[name].loc[common])
    rows.append({'modelo': name.upper(), 'WA média': t['WA (tuned)'].mean(), 'WA std': t['WA (tuned)'].std(), **{f'recall {c}': r['recall'][c] for c in CLASSES}})
rows.append({'modelo': 'Ensemble', 'WA média': table.WA.mean(), 'WA std': table.WA.std(), **{f'recall {c}': rep['recall'][c] for c in CLASSES}})
pd.DataFrame(rows).set_index('modelo').round(3)